# LoveDA training on Colab

This notebook runs the whole pipeline on a free GPU. It gets the code, installs the extra dependencies, downloads LoveDA, trains DeepLabV3 ResNet50, then evaluates, analyzes and shows the results.

Before running, switch the runtime to a GPU. Use the menu Runtime then Change runtime type then pick a GPU such as T4.

Colab sessions are temporary, so the last cell copies your run to Google Drive. Do that or you lose the checkpoint when the session ends.

## 1. Check the GPU

In [ ]:
import torch
print("torch", torch.__version__, "cuda available", torch.cuda.is_available())
!nvidia-smi -L

## 2. Get the code

Two ways. Clone from GitHub if you pushed the project, or upload a zip of the project folder. Run only one of the two cells below.

In [ ]:
# Option A. Clone from GitHub. Replace the URL with your own repository.
REPO_URL = "https://github.com/YOUR_USERNAME/semantic-segmentation-feature.git"
!git clone $REPO_URL
%cd semantic-segmentation-feature

In [ ]:
# Option B. Upload a zip of the project instead of cloning, then unzip it.
# from google.colab import files
# uploaded = files.upload()            # choose your project zip
# import zipfile, io
# name = next(iter(uploaded))
# with zipfile.ZipFile(io.BytesIO(uploaded[name])) as archive:
#     archive.extractall(".")
# %cd semantic-segmentation-feature

## 3. Install the extra dependencies

Colab already provides a CUDA build of torch and torchvision, so do not reinstall those. Installing the pinned CPU torch from requirements would replace the GPU build and slow everything down, so only the lighter extras are installed here.

In [ ]:
!pip -q install seaborn tensorboard pyyaml tqdm scikit-learn gradio

## 4. Download LoveDA

This pulls Train and Val from the official Zenodo record and arranges them. It is a few gigabytes, so it takes a few minutes.

In [ ]:
!python scripts/download_loveda.py --root data/raw/LoveDA

## 5. Train on the GPU

This uses the LoveDA config. The overrides give a shorter first run that still produces a usable model. Raise train.epochs for a stronger result. On a T4 each epoch takes a few minutes.

In [ ]:
!python scripts/train.py --config configs/default.yaml --set train.epochs=15 dataloader.batch_size=8 dataloader.num_workers=2

## 6. Evaluate and analyze on the validation split

LoveDA Test labels are withheld, so the validation split is the held out set.

In [ ]:
!python scripts/evaluate.py --checkpoint results/loveda_deeplabv3/checkpoints/best.pth --split val
!python scripts/analyze.py --checkpoint results/loveda_deeplabv3/checkpoints/best.pth --split val

## 7. View the results inline

In [ ]:
import glob
from IPython.display import Image as IPImage, Markdown, display
base = "results/loveda_deeplabv3/analysis"
display(Markdown(open(base + "/report.md").read()))
for path in sorted(glob.glob(base + "/*.png")) + sorted(glob.glob(base + "/error_maps/*.png")):
    print(path)
    display(IPImage(path))

## 8. Training curves and sample images in TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir results/loveda_deeplabv3/tensorboard

## 9. Interactive prediction app

Launch the upload GUI with a public link. Open the printed gradio.live URL, upload an aerial tile and see the real prediction. Stop the cell to shut the app down.

In [ ]:
!python scripts/app.py --checkpoint results/loveda_deeplabv3/checkpoints/best.pth --share

## 10. Save the run to Google Drive

Colab sessions are temporary. Run this to keep the checkpoint, report and figures.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!cp -r results/loveda_deeplabv3 "/content/drive/MyDrive/loveda_deeplabv3"
print("copied run to Google Drive")